# 01. KV450 inference

KV450 xi+/- likelihood (14 parameters), MAP, MCMC and nested sampling, with the emulated linear transfer function (`data/emulator`). Part A: quick CLI demos sized for the
free 2-vCPU runtime (minutes). Part B: the full chain `scripts/run_kv450_*.py` in the order of `docs/RESULTS.md` section 6, as resumable stages.
The reference results of this analysis are shipped in the repository's `runs/kv450_final/`, so notebook 02 and the plots work without running Part B.

**Acknowledgement.** The DR5 / KiDS-Legacy data are KiDS products. Acknowledge KiDS and cite Wright et al. 2024 (A&A 686, A170) and the KiDS-Legacy papers
(<https://kids.strw.leidenuniv.nl/DR5/legacy_wl.php>); the repository's `README.md` has the full text. The repository is private; do not share the data or results
outside the terms of the KiDS release.

### Which runtime?

| | free Colab | Colab Pro / high-RAM |
|---|---|---|
| vCPU | 2 | about 8 |
| RAM | 12.7 GB | about 51 GB |
| session length | about 12 h (idle and background limits apply) | up to about 24 h |
| local disk | about 100 GB (wiped at the end of the session) | the same |
| Google Drive | persistent, shared 15 GB quota (free) | persistent |

The notebooks use `os.cpu_count()` for threads and worker processes. All times in these notebooks are **estimates** scaled from timings recorded on a
22-core development workstation (WSL2): linear scaling in cores is assumed and a Colab vCPU is assumed to be as fast as one of its cores, which is probably
optimistic. Treat them as lower bounds on a shared, noisy Colab host.

### One-time preparation: access to the private repository

The repository is private, so the notebook needs a GitHub **fine-grained personal access token**. The token is stored in **Colab Secrets**
and is never written into a cell, a file or the output.

1. GitHub, Settings, Developer settings, Personal access tokens, **Fine-grained tokens**, Generate new token. Resource owner: the account that owns the
   repository. Repository access: **Only select repositories**, choose this repository. Permissions: **Contents: Read-only** (Metadata is added
   automatically). Pick a short expiry. Copy the token once.
2. In Colab, left sidebar, the key icon (**Secrets**), Add new secret: name `GH_TOKEN`, value = the token, switch **Notebook access** on for this notebook
   (the switch is per notebook).
3. Edit `GITHUB_REPO` in the setup cell below (`owner/name`).

No GitHub access? Put a zip of the repository at `MyDrive/kidspipe/kidspipe_repo.zip`; the setup cell unpacks it instead of cloning.

## Session setup

A Colab runtime is fresh in every session: **run the next cell first, every time**. It mounts Drive, clones or updates the repository, installs the package if needed and sets the environment variables (`KIDSPIPE_RUNS`, `KIDSPIPE_DR5_WORK`, `KIDSPIPE_DR5_CATALOGUE`, `KIDSPIPE_PLOTS`, thread counts, numba cache on Drive) described in `docs/COLAB.md`.

In [ ]:
#@title 1. Session setup (run this first, in EVERY session) { display-mode: "form" }
#@markdown Private repository to clone (`owner/name`); the token comes from the Colab Secret `GH_TOKEN`.
GITHUB_REPO = "OWNER/REPO"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown Persistent outputs live on Drive; the 16 GB catalogue lives on the local disk (never on Drive).
DRIVE_ROOT = "/content/drive/MyDrive/kidspipe"  #@param {type:"string"}
CATALOGUE_DIR = "/content/dr5"  #@param {type:"string"}
REPO_DIR = "/content/kidspipe"

import base64, importlib, os, pathlib, shutil, subprocess, sys

def _git(args, token=None, cwd=None):
    """git, with the token passed as an HTTP header (not stored in .git/config); the token is masked in anything returned."""
    cmd, secret = ["git"], []
    if token:
        b64 = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        cmd += ["-c", f"http.extraheader=Authorization: Basic {b64}"]
        secret = [token, b64]
    r = subprocess.run(cmd + args, cwd=cwd, capture_output=True, text=True)
    out = r.stdout + r.stderr
    for s in secret:
        out = out.replace(s, "***")
    return r.returncode, out.strip()

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: Drive is not mounted; DRIVE_ROOT must be a writable directory.")

token = None
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception as e:                                   # secret missing, or notebook access not granted
    print(f"No GH_TOKEN secret available ({type(e).__name__}); using an existing clone or the Drive zip if there is one.")

repo = pathlib.Path(REPO_DIR)
url = f"https://github.com/{GITHUB_REPO}.git"
if (repo / "pyproject.toml").exists():
    rc, out = _git(["fetch", "--depth", "1", "origin", BRANCH], token, cwd=repo)
    if rc == 0:
        _git(["reset", "--hard", "FETCH_HEAD"], cwd=repo)
        print("repository updated to the latest", BRANCH)
    else:
        print("could not update (using the existing clone):", out[-300:])
elif token or not pathlib.Path(DRIVE_ROOT, "kidspipe_repo.zip").exists():
    if GITHUB_REPO.startswith("OWNER/"):
        raise SystemExit("Edit GITHUB_REPO (owner/name) in this cell, and add the GH_TOKEN secret (see the markdown above).")
    rc, out = _git(["clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR], token)
    if rc != 0:
        raise SystemExit("git clone failed (wrong GITHUB_REPO, or the token lacks Contents: read access?):\n" + out[-500:])
    print("cloned", GITHUB_REPO)
else:
    tmp = pathlib.Path("/content/_unzip"); shutil.rmtree(tmp, ignore_errors=True)
    shutil.unpack_archive(str(pathlib.Path(DRIVE_ROOT, "kidspipe_repo.zip")), str(tmp))
    top = [p for p in tmp.iterdir() if p.is_dir()]
    shutil.move(str(top[0]) if len(top) == 1 and not (tmp / "pyproject.toml").exists() else str(tmp), REPO_DIR)
    print("unpacked the repository from the Drive zip")

sys.path.insert(0, f"{REPO_DIR}/scripts/colab")
import setup_env; importlib.reload(setup_env)
INFO = setup_env.setup(REPO_DIR, DRIVE_ROOT, CATALOGUE_DIR)      # env vars, pip install -e '.[all]' if needed, versions
os.chdir(REPO_DIR)
WORKERS = os.cpu_count() or 1

## Part A. Quick demos (free runtime)

Outputs go to `$KIDSPIPE_RUNS/<name>/` on Drive. RAM is not an issue (about 310 MB per sampler worker).

In [ ]:
import os, json, pathlib
RUNS = pathlib.Path(os.environ["KIDSPIPE_RUNS"])
def show(path, keys):
    d = json.load(open(path)); print({k: d[k] for k in keys if k in d})

**A1. Likelihood at the spec test cosmology** (chi2, ln L, S8). Seconds.

In [ ]:
!python -m kidspipe loglike --name colab_loglike --json

**A2. MAP search, small** (2 starts, 40 iterations, one round; about 10-25 min on 2 vCPU, estimate; the documented size of this demo took 275 s + 66 s Hessian on 22 loaded cores).

In [ ]:
!python -m kidspipe map --starts 2 --maxiter 40 --rounds 1 --polish none --workers {min(WORKERS, 2)} --worker-threads 1 --name colab_map
show(RUNS / "colab_map" / "map.json", ["chi2"])

**A3. Blocked-Metropolis MCMC, tiny** (2 chains, 100 burn-in + 200 cycles; minutes). Not converged, only a pipeline check.

In [ ]:
!python -m kidspipe mcmc --map {RUNS}/colab_map/map.json --chains 2 --workers {min(WORKERS, 2)} --worker-threads 1 --burn 100 --cycles 200 --name colab_mcmc

**A4. Nested sampling smoke test** (nautilus, 50 live points, at most 300 likelihood calls; a few minutes).

In [ ]:
!python -m kidspipe nested --sampler nautilus --nlive 50 --n-eff 100 --max-calls 300 --workers {WORKERS} --name colab_nested

**A5. Timing per parameter block on this machine** (about a minute).

In [ ]:
!python -m kidspipe bench --workers 1 --n-slow 3 --n-medium 10 --n-fast 100 --name colab_bench

## Part B. The full KV450 chain

### Expected wall times of the full chain (ESTIMATES)

Core-hours are from the logs on the development workstation; wall = core-hours / min(processes, vCPU). Assumes a Colab vCPU as fast as a workstation core.

| step (script) | core-h | processes | wall, 2 vCPU | wall, 8 vCPU |
|---|---|---|---|---|
| MAP explore (`run_kv450_map.py --stage explore --starts 36`) | 6.3 | 13 | 3.2 h | 0.8 h |
| MAP polish, L-BFGS (`--top 6 --polish none`) | 0.1 | 6 | 5 min | 2 min |
| MAP polish, Powell (`--top 3`) | 1.5 | 3 | 0.75 h | 0.5 h |
| `run_kv450_mapbest`, `_exact map`, `_chi2split` | small | 1 | 10 min | 10 min |
| profile (`run_kv450_profile.py`) | 1-2 (not logged) | 12 | 0.5-1 h | 0.2 h |
| pilot chains (`run_kv450_mcmc.py pilot`) | 1.8 | 6 | 0.9 h | 0.3 h |
| production 1 (6 chains x 80 min of CPU) | 8 | 6 | 4 h | 1.3 h |
| production 2 (8 chains x 110 min of CPU) | 14.7 | 8 | 7.3 h | 1.8 h |
| analyze, exact-CAMB check of 200 samples, maximum likelihood | about 1 | 1-6 | 0.5 h | 0.3 h |
| nested sampling (`nautilus2`, `nautilus3`) | 2.2 + 3.2 | 8 | 1.1 + 1.6 h | 0.3 + 0.4 h |
| sensitivity variants (`run_kv450_sens.py`) | about 7 | 3 | 3.3 h | 2.2 h |
| re-weighting (`run_kv450_reweight.py`) | 0.5-1 | 6 | 0.5 h | 0.2 h |
| **total** | about 45 | | **about 24 h (3 sessions)** | **about 9 h** |

The `quick` profile below runs the same stages at toy size (a few chains of 10 minutes): about 2-3 h on 2 vCPU, to check that the whole chain works. Its results go to
`MyDrive/kidspipe/runs_quick` and are not the published numbers.

**Resumability.** Each stage is skipped when its marker `<runs>/stages/<name>.done` exists, so after a lost session you re-run the setup cell and the same stage cell. Inside a
stage: MAP stages and nested sampling (nautilus checkpoint) resume; **`run_kv450_mcmc.py prod` does not** (it writes `prod_c*.npz` checkpoints every 5 minutes but a restart tunes
and samples again). A production stage is therefore one uninterrupted block of about 4 h (prod1) or 7 h (prod2) on 2 vCPU: start it with at least that much session left.

In [ ]:
#@title Profile and helpers { display-mode: "form" }
PROFILE = "quick"  #@param ["quick", "documented"]
import os, pathlib
from setup_env import stage
W = WORKERS
w = lambda n: max(1, min(n, W))                      # worker processes for a step that can use n at most
if PROFILE == "quick":
    os.environ["KIDSPIPE_RUNS"] = f"{DRIVE_ROOT}/runs_quick"     # keep toy results apart from the real ones
else:
    os.environ["KIDSPIPE_RUNS"] = f"{DRIVE_ROOT}/runs"
RUNS = pathlib.Path(os.environ["KIDSPIPE_RUNS"]); OUT = RUNS / "kv450_final"; OUT.mkdir(parents=True, exist_ok=True)
S = "python3 scripts"
mins = lambda m, chains: round(m * chains / w(chains))          # --minutes is WALL time: keep the CPU time per chain of the documented run
print(f"profile {PROFILE}: {W} vCPU, outputs in {OUT}")

### B1. Global MAP (resumable stages)

In [ ]:
if PROFILE == "documented":
    stage("map_explore",  f"{S}/run_kv450_map.py --stage explore --starts 36 --workers {w(13)}")
    stage("map_polish_lbfgs", f"{S}/run_kv450_map.py --stage polish --top 6 --workers {w(6)} --polish none && cp {OUT}/map_starts.json {OUT}/map_polish_lbfgs.json")
    stage("map_polish_powell", f"{S}/run_kv450_map.py --stage polish --top 3 --workers {w(3)} --polish powell")
else:
    stage("map_explore",  f"{S}/run_kv450_map.py --stage explore --starts 6 --workers {w(6)}")
    stage("map_polish_lbfgs", f"{S}/run_kv450_map.py --stage polish --top 2 --workers {w(2)} --polish none && cp {OUT}/map_starts.json {OUT}/map_polish_lbfgs.json")
    stage("map_polish_powell", f"{S}/run_kv450_map.py --stage polish --top 1 --workers 1 --polish powell")
stage("map_best",  f"{S}/run_kv450_mapbest.py")
stage("map_exact", f"{S}/run_kv450_exact.py map")
stage("chi2_split", f"{S}/run_kv450_chi2split.py")
if PROFILE == "documented":
    stage("profile", f"{S}/run_kv450_profile.py --workers {w(12)}")

### B2. Posterior: pilot, two production runs, analysis

Not resumable inside `prod` (see above). `--minutes` is wall time per chain: the cell scales it by chains / processes so every chain keeps the CPU time of the documented run.
Check `Runtime, Change runtime type` first if you want the 8-vCPU machine.

In [ ]:
if PROFILE == "documented":
    pilot = "--chains 6 --burn 1500 --cycles 1500"; prod1 = f"--chains 6 --minutes {mins(80, 6)} --tune 600 --seed 21"; prod2 = f"--chains 8 --minutes {mins(110, 8)} --tune 600 --seed 22"
else:
    pilot = "--chains 2 --burn 300 --cycles 300"; prod1 = "--chains 2 --minutes 10 --tune 200 --seed 21"; prod2 = "--chains 2 --minutes 10 --tune 200 --seed 22"
stage("pilot", f"{S}/run_kv450_mcmc.py pilot --tag pilot1 {pilot} --starts {OUT}/starts.json")
stage("prod1", f"{S}/run_kv450_mcmc.py prod --tag prod1 --pilot pilot1 {prod1}")
stage("prod2", f"{S}/run_kv450_mcmc.py prod --tag prod2 --pilot pilot1 {prod2}")
stage("analyze", f"{S}/run_kv450_analyze.py prod1,prod2 --burn-frac 0.05")
stage("exact_chain", f"{S}/run_kv450_exact.py chain prod1,prod2 200")
if PROFILE == "documented":
    stage("ml", f"{S}/run_kv450_ml.py --workers {w(6)}")                # needs the production chains (despite its position in RESULTS.md)

### B3. Nested sampling (nautilus; checkpointed, so an interrupted stage resumes)

In [ ]:
if PROFILE == "documented":
    stage("nautilus2", f"{S}/run_kv450_nested.py --workers {w(8)} --z-grid reduced --n-live 2000 --n-eff 8000 --tag nautilus2 --seed 12")
    stage("nautilus3", f"{S}/run_kv450_nested.py --workers {w(8)} --z-grid spec --n-live 2000 --n-eff 8000 --tag nautilus3 --seed 13")
else:
    stage("nautilus3", f"{S}/run_kv450_nested.py --workers {w(8)} --z-grid reduced --n-live 200 --n-eff 500 --tag nautilus3 --seed 13")
stage("report", f"{S}/run_kv450_report.py > {OUT}/report.txt && tail -n 40 {OUT}/report.txt")   # tables of docs/RESULTS.md sections 2.4 and 3

### B4. Systematic sensitivity and re-weighting

In [ ]:
if PROFILE == "documented":
    stage("sens_main", f"{S}/run_kv450_sens.py --variants abar2,abar313,aia0,nomcorr,baseline,reduced,kcap --workers {w(3)} --cycles 4000 --burn 1000 --chains 3 --map-file map_best.json "
                       f"&& cp {OUT}/sens/sens_summary.json {OUT}/sens/sens_summary_main.json")     # run_kv450_sens.py overwrites sens_summary.json on every call
    stage("sens_camb", f"{S}/run_kv450_sens.py --variants camb_mead --no-chain --workers 1 --map-file map_best.json")
    stage("reweight", f"{S}/run_kv450_reweight.py --tags prod1,prod2 --variants kcap,reduced,camb_mead --n 500 --workers {w(6)}")
else:
    stage("sens_main", f"{S}/run_kv450_sens.py --variants baseline,reduced --workers {w(2)} --cycles 600 --burn 200 --chains 2 --map-file map_best.json")
    stage("reweight", f"{S}/run_kv450_reweight.py --tags prod1,prod2 --variants kcap,reduced --n 100 --workers {w(2)}")

### B5. Compare with the reference results shipped in the repository

The documented run reproduces the shipped numbers up to Monte-Carlo noise (different chains); the `quick` profile will not match closely.

In [ ]:
import json
def s8(p):
    d = json.load(open(p))["stats"]["S8"]; return f"S8 = {d['mean']:.4f} +- {d['std']:.4f}"
mine = OUT / "combined_prod1_prod2" / "analysis.json"
ref = pathlib.Path(REPO_DIR) / "runs" / "kv450_final" / "combined_prod1_prod2" / "analysis.json"
print("yours   :", s8(mine) if mine.exists() else "(not produced yet)")
print("shipped :", s8(ref))
print("MAP chi2 yours  :", json.load(open(OUT / "map_best.json"))["chi2"] if (OUT / "map_best.json").exists() else "-")
print("MAP chi2 shipped:", json.load(open(pathlib.Path(REPO_DIR) / "runs/kv450_final/map_best.json"))["chi2"])

### Notes

* If a stage fails or the session dies, fix the cause, re-run the setup cell, then the same stage cell: finished stages are skipped, `force` is available as `stage(name, cmd, force=True)`.
* `$KIDSPIPE_RUNS` on Drive is read first by every later step, the shipped `runs/` second: a step you did not re-run falls back to the shipped reference result. In `documented`
  mode run the steps in order; in an empty `runs_quick` the missing earlier steps come from the shipped results.
* Free tier: keep the browser tab open; long cells are cut when the runtime is reclaimed (idle or maximum duration). Stage markers make that harmless except inside `prod`.